# Czysty notatnik do generowania działającego (miejmy nadzieję) modelu detekcji zagrożeń na podstawie logów Wazuh
## ✨Organicznie uprawiany kod✨
Zalecam korzystanie z menedżera środowisk python `uv`, bo jest lepszy i nie tak ociężały jak `conda` czy domyślny `venv`.
Tworzenie środowiska
```
uv venv .venv --python cp3.14 # argument --python opcjonalnie cp3.14 po prostu jest szybki ;)
```
Aktywacja środowiska
```
source .venv/bin/activate     # to pewnie będzie inaczej na Windows
```
Instalacja bibliotek
```
uv pip install ipykernel pandas numpy
```

## Połączony potok: okna 10 s + historia 60 s (Plik 1) na czystych cechach bez wycieków (Plik 2)

1. **Filtracja źródeł**: wczytujemy wyłącznie alerty Wazuh (`*_wazuh.json`), odrzucając wpisy Suricaty oraz AMinera.
2. **Ekstrakcja czystych cech per zdarzenie (bez reguł, IP i kalendarza)**:
   - anonimizacja adresów IP (`srcip`/`dstip` $\rightarrow$ klasa sieci prywatna/zewnętrzna; surowy `srcip` służy wyłącznie do policzenia unikalnych źródeł w oknie 10 s i jest usuwany),
   - wyznaczenie roli hosta (`WWW`, `mail`, `dns`, `auth`, `other`) z `location` oraz kategorii portu z `data.dstport`,
   - metryki strukturalne i flagi `data.url` (entropia, długość, głębokość ścieżki, zapytania, flagi podatności),
   - maskowanie IP, domen (`.biz`, `.com` itd.), nazw plików i znaczników czasu w `full_log` oraz wyszukiwanie wzorców narzędzi (`nmap`, `dirb`, `wpscan` itp.) i słów kluczowych bez rozróżniania wielkości liter (`lowercase`).
3. **Agregacja w okna 10 s i odrzucenie okien granicznych**: grupowanie zdarzeń w przedziały `10s`, wykluczenie okien przecinających granice z `labels.csv` ze zbioru uczącego przy jednoczesnym zachowaniu ich w pełnej osi czasu.
4. **Sekwencja 60 s (6 kroków × 10 s)**: budowa tensora `(N, 6, F)` z transformacją `log1p` dla liczników.

definicje ekstraktorów czystych cech i wczytanie przedziałów labels.csv

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv
import ipaddress
import json
import math
import re
import numpy as np
import pandas as pd

# Ścieżka do danych (dopasuj do katalogu roboczego notatnika: "../data/ait_ads" lub "data/ait_ads")
DATA_DIR = Path("../data/ait_ads") if Path("../data/ait_ads").is_dir() else Path("data/ait_ads")

SCENARIOS = ["fox", "harrison", "russellmitchell", "santos", "shaw", "wardbeck", "wheeler", "wilson"]
TRAIN_SCENARIOS = ["fox", "russellmitchell", "shaw", "wardbeck", "wheeler", "wilson", "santos"]
TEST_SCENARIOS = ["harrison"]

# Mapowanie 10 faz z labels.csv na 7 głównych klas ataków (zgodnie z wymaganiami prowadzącego)
ATTACK_7_MAP = {
    "dirb": "dirb",
    "wpscan": "wpscan",
    "dnsteal": "dnsteal",
    "cracking": "cracking",
    "service_scans": "scanning",
    "network_scans": "scanning",
    "webshell": "exploitation",
    "reverse_shell": "exploitation",
    "privilege_escalation": "privilege_escalation",
    "service_stop": "privilege_escalation",
}

# Prekompilowane wyrażenia regularne (case-insensitive)
RX_IPV4 = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
RX_IPV6 = re.compile(r"\b[0-9a-f]{1,4}(?::[0-9a-f]{1,4}){2,7}\b")
RX_DOMAIN = re.compile(r"\b(?:[a-z0-9-]+\.)+(?:biz|com|net|org|io|info|local|lan|intranet)\b")
RX_FILE = re.compile(r"\b[a-z0-9_.-]+\.(?:php|txt|zip|sql|sh|py|exe|bak|tar|gz|log)\b")
RX_TS = re.compile(r"\b\d{4}-\d{2}-\d{2}[t\s]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:z|[+-]\d{2}:?\d{2})?\b")
RX_TOKEN = re.compile(r"[a-z0-9+/=%_.-]{8,}")
RX_ALNUM = re.compile(r"[a-z0-9]+")

URL_WP_NEEDLES = ("wp-admin", "wp-content", "wp-json", "wp-login", "xmlrpc", "wp-config", "wpscan")
URL_ADMIN_DB_NEEDLES = ("phpmyadmin", "admin", "config", "secret", "pass", "user", "backup", "dump", ".git", "sql")
URL_EXEC_NEEDLES = ("shell", "cmd", "remote", "proc", "include", ".php", ".zip", ".txt", "ocs/")
UA_SCANNER_NEEDLES = ("dirb", "wpscan", "nmap", "ncat", "wget", "headlesschrome")
LOG_AUTH_FAIL_NEEDLES = ("failed", "invalid", "authentication failure", "denied", "unauthorized")
LOG_PRIV_NEEDLES = ("sudo", "su:", "session opened", "chmod", "chown")

_ip_cache: dict[str, int] = {}

def classify_ip(ip_val: object) -> int:
    """Zwraca: 1 = prywatne (wew), 0 = publiczne (zew), -1 = brak/niepoprawne."""
    if not isinstance(ip_val, str) or not ip_val:
        return -1
    clean = ip_val.split(":")[0].strip()
    if clean in _ip_cache:
        return _ip_cache[clean]
    try:
        res = int(ipaddress.ip_address(clean).is_private)
    except ValueError:
        res = -1
    _ip_cache[clean] = res
    return res

def extract_host_role(loc: object) -> str:
    if not isinstance(loc, str):
        return "other"
    l = loc.lower()
    if any(k in l for k in ("apache", "nginx", "access.log", "error.log", "www", "http")):
        return "WWW"
    if any(k in l for k in ("mail", "postfix", "dovecot", "exim", "smtp", "imap")):
        return "mail"
    if any(k in l for k in ("bind", "named", "dns", "unbound")):
        return "dns"
    if any(k in l for k in ("auth", "secure", "sshd", "audit")):
        return "auth"
    return "other"

def port_category(port_val: object) -> str:
    try:
        port = int(float(str(port_val)))
    except (ValueError, TypeError):
        return "other"
    return {80: "http", 443: "https", 53: "dns", 587: "smtp", 5800: "vnc"}.get(port, "other")

def entropy_of(text: str) -> float:
    if not text:
        return 0.0
    counts = Counter(text)
    total = len(text)
    return round(-sum((c / total) * math.log2(c / total) for c in counts.values()), 4)

def mask_and_lower_log(raw: object) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw[:400].lower()  # wszystkie litery są sobie równoważne
    s = RX_IPV4.sub(" ip_token ", s)
    s = RX_IPV6.sub(" ipv6_token ", s)
    s = RX_DOMAIN.sub(" domain_token ", s)
    s = RX_FILE.sub(" file_token ", s)
    s = RX_TS.sub(" ts_token ", s)
    return s

def load_intervals(path: Path) -> dict[str, list[tuple[float, float, str]]]:
    result: dict[str, list[tuple[float, float, str]]] = defaultdict(list)
    with path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            sc = row["scenario"].strip().lower()
            start, end = float(row["start"]), float(row["end"])
            raw_attack = row["attack"].strip()
            mapped_attack = ATTACK_7_MAP.get(raw_attack, raw_attack)
            result[sc].append((start, end, mapped_attack))
    for sc in result:
        result[sc].sort()
    return result

intervals = load_intervals(DATA_DIR / "labels.csv")
print("Wczytano przedziały z labels.csv dla scenariuszy:", list(intervals.keys()))

wczytanie logów Wazuh, ekstrakcja czystych cech i etykietowanie czasu

In [ ]:
rows = []
for sc in SCENARIOS:
    path = DATA_DIR / f"{sc}_wazuh.json"
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            alert = json.loads(line)

            # 1. Odrzucenie Suricaty i AMinera (tylko czysty Wazuh)
            location = str(alert.get("location") or "")
            rule = alert.get("rule") or {}
            decoder = alert.get("decoder") or {}
            rule_desc = str(rule.get("description") or "").lower()
            rule_groups = [str(g).lower() for g in (rule.get("groups") or [])]
            decoder_name = str(decoder.get("name") or "").lower()

            if (
                "suricata" in location.lower()
                or "suricata" in rule_desc
                or "suricata" in rule_groups
                or "aminer" in location.lower()
                or "aminer" in decoder_name
                or "aminer" in rule_groups
            ):
                continue

            data = alert.get("data") or {}
            srcip_raw = str(data.get("srcip") or "").strip()
            dstip_raw = str(data.get("dstip") or "").strip()
            src_cls = classify_ip(srcip_raw)
            dst_cls = classify_ip(dstip_raw)

            url_raw = str(data.get("url") or "")
            url_low = url_raw.lower()
            log_masked = mask_and_lower_log(alert.get("full_log"))

            dstuser = str(data.get("dstuser") or "").strip()
            srcuser = str(data.get("srcuser") or "").strip()
            proto = str(data.get("protocol") or "").upper()

            rows.append({
                "scenario": sc,
                "timestamp": alert.get("@timestamp"),
                # Tymczasowe pole tylko do zliczenia unikalnych źródeł w oknie 10s (nie trafia do modelu)
                "srcip_anon": srcip_raw,
                # Zanonimizowane cechy sieciowe i roli hosta
                "src_private": int(src_cls == 1),
                "src_external": int(src_cls == 0),
                "dst_private": int(dst_cls == 1),
                "dst_external": int(dst_cls == 0),
                "host_role": extract_host_role(location),
                "dstport_cat": port_category(data.get("dstport")),
                # Tożsamość i protokół (bez nazw użytkowników)
                "is_root": int(dstuser == "root"),
                "has_user": int(bool(dstuser or srcuser)),
                "is_get": int(proto == "GET"),
                "is_post": int(proto == "POST"),
                "is_head": int(proto == "HEAD"),
                # Cechy strukturalne URL
                "url_len": float(len(url_raw)),
                "url_entropy": entropy_of(url_low),
                "url_parts": float(len(url_low.split("/"))) if url_low else 0.0,
                "url_query": int("?" in url_low),
                "url_encoded": float(url_low.count("%")),
                "url_max_token": float(max((len(t) for t in RX_TOKEN.findall(url_low)), default=0)),
                # Flagi wzorców URL oraz narzędzi skanujących (z zamaskowanego logu lowercase)
                "flag_wp": int(any(n in url_low for n in URL_WP_NEEDLES)),
                "flag_admin_db": int(any(n in url_low for n in URL_ADMIN_DB_NEEDLES)),
                "flag_exec_file": int(any(n in url_low for n in URL_EXEC_NEEDLES)),
                "flag_ua_scanner": int(any(n in log_masked for n in UA_SCANNER_NEEDLES)),
                # Cechy leksykalne z zamaskowanego full_log (lowercase)
                "log_auth_fail": int(any(n in log_masked for n in LOG_AUTH_FAIL_NEEDLES)),
                "log_priv_cmd": int(any(n in log_masked for n in LOG_PRIV_NEEDLES)),
                "log_has_domain": int("domain_token" in log_masked),
                "log_has_file": int("file_token" in log_masked),
            })

df_events = pd.DataFrame(rows)
df_events["event_time"] = pd.to_datetime(df_events["timestamp"], utc=True)
times = ((df_events["event_time"] - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(seconds=1)).to_numpy()

# Przypisanie etykiet czasowych z labels.csv w obrębie każdego scenariusza
scenarios_arr = df_events["scenario"].to_numpy()
labels = np.full(len(df_events), "background", dtype=object)
match_count = np.zeros(len(df_events), dtype=np.uint8)

for sc, phase_intervals in intervals.items():
    sc_mask = scenarios_arr == sc
    for start, end, phase in phase_intervals:
        in_phase = sc_mask & (times >= start) & (times < end)
        match_count[in_phase] += 1
        labels[in_phase] = phase

assert not (match_count > 1).any(), "Nachodzące etykiety — sprawdź labels.csv"
df_events["attack_phase"] = labels
df_events = df_events.drop(columns=["timestamp"])

print("Wczytano czystych zdarzeń Wazuh:", len(df_events))
display(pd.crosstab(df_events["scenario"], df_events["attack_phase"]))

szkielet okien 10 s z Pliku 1 + odrzucenie okien granicznych

In [ ]:
WINDOW_SECONDS = 10

df_events["window_start"] = df_events["event_time"].dt.floor(f"{WINDOW_SECONDS}s")
keys = ["scenario", "window_start"]

# 1. Agregacja czystych cech numerycznych i zliczeń w oknie 10-sekundowym
windows = df_events.groupby(keys).agg(
    alert_count=("attack_phase", "size"),
    unique_srcips=("srcip_anon", lambda s: s[s != ""].nunique()),
    src_private_cnt=("src_private", "sum"),
    src_external_cnt=("src_external", "sum"),
    dst_private_cnt=("dst_private", "sum"),
    dst_external_cnt=("dst_external", "sum"),
    root_user_cnt=("is_root", "sum"),
    known_user_cnt=("has_user", "sum"),
    http_get_cnt=("is_get", "sum"),
    http_post_cnt=("is_post", "sum"),
    http_head_cnt=("is_head", "sum"),
    mean_url_len=("url_len", "mean"),
    max_url_len=("url_len", "max"),
    mean_url_entropy=("url_entropy", "mean"),
    max_url_entropy=("url_entropy", "max"),
    mean_url_parts=("url_parts", "mean"),
    url_query_cnt=("url_query", "sum"),
    max_url_encoded=("url_encoded", "max"),
    max_url_token=("url_max_token", "max"),
    flag_wp_cnt=("flag_wp", "sum"),
    flag_admin_db_cnt=("flag_admin_db", "sum"),
    flag_exec_file_cnt=("flag_exec_file", "sum"),
    flag_ua_scanner_cnt=("flag_ua_scanner", "sum"),
    log_auth_fail_cnt=("log_auth_fail", "sum"),
    log_priv_cmd_cnt=("log_priv_cmd", "sum"),
    log_domain_cnt=("log_has_domain", "sum"),
    log_file_cnt=("log_has_file", "sum"),
    label_count=("attack_phase", "nunique"),
    attack_phase=("attack_phase", "first"),
)

# 2. Rozbicie ról hostów (WWW, mail, dns, auth, other) i portów na liczniki w oknie 10s
role_counts = df_events.groupby(keys + ["host_role"]).size().unstack(fill_value=0)
role_counts.columns = [f"role_{c}_cnt" for c in role_counts.columns]

port_counts = df_events.groupby(keys + ["dstport_cat"]).size().unstack(fill_value=0)
port_counts.columns = [f"port_{c}_cnt" for c in port_counts.columns]

# Zachowujemy pełną tablicę wszystkich okien (również granicznych) do budowy ciągłej historii 60s
all_windows_full = windows.join(role_counts).join(port_counts)

# 3. Wykluczenie okien przecinających granice faz z labels.csv (dokładnie jak w Pliku 1)
windows_reset = all_windows_full.reset_index()
boundary_crossing = np.zeros(len(windows_reset), dtype=bool)

for sc, phase_intervals in intervals.items():
    mask = windows_reset["scenario"].eq(sc).to_numpy()
    positions = np.flatnonzero(mask)
    starts = (
        (windows_reset.loc[mask, "window_start"] - pd.Timestamp("1970-01-01", tz="UTC"))
        / pd.Timedelta(seconds=1)
    ).to_numpy()

    crosses = np.zeros(len(starts), dtype=bool)
    for attack_start, attack_end, _ in phase_intervals:
        for boundary in (attack_start, attack_end):
            crosses |= (starts < boundary) & (boundary < starts + WINDOW_SECONDS)
    boundary_crossing[positions] = crosses

invalid = boundary_crossing | windows_reset["label_count"].gt(1).to_numpy()
df_windows = windows_reset.loc[~invalid].drop(columns=["label_count"]).reset_index(drop=True)

print(f"Wszystkie okna 10s: {len(windows_reset)} | Odrzucone graniczne: {int(invalid.sum())} | Czyste okna: {len(df_windows)}")
display(pd.crosstab(df_windows["scenario"], df_windows["attack_phase"]))

budowa sekwencji 60 s (N, 6, F) i skalowanie log1p

In [ ]:
SEQ_LEN = 6        # 6 kroków × 10 s = 60 sekund kontekstu historycznego
STEP_SECONDS = 10

non_feature_cols = {"scenario", "window_start", "label_count", "attack_phase"}
sequence_features = [c for c in all_windows_full.columns if c not in non_feature_cols]

# Cechy ciągłe (średnie/maksima) nie przechodzą log1p, wszystkie liczniki zdarzeń przechodzą log1p
continuous_cols = {"mean_url_len", "max_url_len", "mean_url_entropy", "max_url_entropy", "mean_url_parts"}
log1p_indices = [i for i, col in enumerate(sequence_features) if col not in continuous_cols]

# Pełna oś czasu okien (włącznie z granicznymi, aby historia 60s nie miała sztucznych dziur)
all_windows_matrix = all_windows_full.reindex(columns=sequence_features, fill_value=0).astype("float32")

X_sequences = np.zeros((len(df_windows), SEQ_LEN, len(sequence_features)), dtype=np.float32)
for step in range(SEQ_LEN):
    lag = (SEQ_LEN - 1 - step) * STEP_SECONDS
    requested_index = pd.MultiIndex.from_arrays(
        [df_windows["scenario"], df_windows["window_start"] - pd.Timedelta(seconds=lag)],
        names=["scenario", "window_start"],
    )
    X_sequences[:, step, :] = all_windows_matrix.reindex(
        requested_index, fill_value=0.0
    ).to_numpy(dtype=np.float32)

# Skalowanie log1p dla wszystkich kolumn zliczeniowych
X_sequences[:, :, log1p_indices] = np.log1p(np.maximum(X_sequences[:, :, log1p_indices], 0.0))
assert np.isfinite(X_sequences).all()

# Podział na zbiór treningowy i testowy po scenariuszach
train_mask = df_windows["scenario"].isin(TRAIN_SCENARIOS).to_numpy()
test_mask = df_windows["scenario"].isin(TEST_SCENARIOS).to_numpy()

X_train_seq = X_sequences[train_mask]
y_train = df_windows.loc[train_mask, "attack_phase"].to_numpy()

X_test_seq = X_sequences[test_mask]
y_test = df_windows.loc[test_mask, "attack_phase"].to_numpy()

print(f"Gotowy tensor sekwencji X_train_seq: {X_train_seq.shape} (okna, 6 kroków po 10s, {len(sequence_features)} czystych cech)")
print(f"Gotowy tensor sekwencji X_test_seq : {X_test_seq.shape}")